In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer


In [2]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, size_ix, lat_ix, lng_ix = 0, 1, 4, 5

class CustomFeaturesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, center_lat=41.3171, center_lng=69.2797):
        # Initialize center coordinates
        self.center_lat = center_lat
        self.center_lng = center_lng
        
    def fit(self, X, y=None):
        return self # Transformer only transforms, does not learn anything
        
    def transform(self, X):
        X = np.asarray(X)
        
        # 1. Calculate room_size (size / rooms)
        room_size = X[:, size_ix] / X[:, rooms_ix]
        
        # 2. Calculate distance_to_center (Haversine)
        R = 6371 # Earth radius
        
        # Take lat and lng columns from X array and convert to radians
        lat1, lon1 = map(np.radians, [X[:, lat_ix], X[:, lng_ix]])
        lat2, lon2 = map(np.radians, [self.center_lat, self.center_lng])
        
        dlon = lon2 - lon1 
        dlat = lat2 - lat1 
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a)) 
        distance_to_center = R * c
        
        # Add two new columns to the original X array (using np.c_)
        return np.c_[X, room_size, distance_to_center]

In [3]:
df = pd.read_csv("../data/processed/tashkent_houses.csv")
df.head()

,district,rooms,size,level,max_levels,price,lat,lng
0,Yunusobod,3,57.0,4,4,52000,41.371471,69.281049
1,Yakkasaroy,2,52.0,4,5,56000,41.291115,69.261104
2,Chilonzor,2,42.0,4,4,37000,41.280784,69.223683
3,Chilonzor,3,65.0,1,4,49500,41.290163,69.196862
4,Chilonzor,3,70.0,3,5,55000,41.300156,69.210831


In [4]:
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=df['district'])
X_train_num = X_train.drop('district', axis=1)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(5924, 7) (5924,)
(1481, 7) (1481,)


In [5]:
numeric_attr =  list(X_train_num)
categorical_attr = ['district']

In [6]:
num_pipeline = Pipeline([
    ('custom_features_adder', CustomFeaturesAdder()),
    ('std_scaler', StandardScaler()) 
])

num_pipeline.fit_transform(X_train_num.values)[:,1]

array([ 1.35526899, -0.07444581,  0.26868574, ..., -0.78930321,
       -0.36038877, -0.70352032], shape=(5924,))

In [ ]:
full_pipeline = ColumnTransformer([
    ('num', num_pipeline, numeric_attr),
    ('cat', OneHotEncoder(), categorical_attr)
])

In [ ]:
jj = full_pipeline.fit_transform(X_train)
jj.shape


(5924, 20)